# Interactive docking box (DDOS-6928 / DDOS-6935)

Demo of production `Docking.show_box(interactive=True)` — rotate the search box in
molstar, click **Apply to notebook**, and read `rotation_deg` on the `Docking`
instance.

## How to run

```bash
make demo-interactive-docking-box
```

This syncs `dev` + `core` + `tools` into `.venv`, registers the **Python (do-dd-client)**
kernel, and opens JupyterLab in your browser. Select that kernel if prompted.

The interactive viewer uses bundled BRD fixtures (no platform sync). The **Run
rotated docking** section at the bottom calls the platform — you need `deeporigin login`,
a billing-enabled org, and toolbox docking **3.4.32+** deployed.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

## Kernel check

Run the cell below first. If it errors, your notebook kernel is **not** the project
`.venv` — widgets will show as text (`<_IframeCommWidget object at ...>`) instead of
rendering. Fix with:

```bash
cd /path/to/do-dd-client
make jupyter
```

Then in the notebook: **Kernel → Change Kernel → Python (do-dd-client)**.

Use **browser JupyterLab** from `make demo-interactive-docking-box` (not a mismatched
interpreter in a multi-root workspace).

In [ ]:
import sys
from pathlib import Path

import anywidget
import ipywidgets


def _find_repo_root() -> Path:
    """Return do-dd-client root from repo root or docs/notebooks/dirty/."""
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "tests").is_dir():
            return candidate
    raise RuntimeError(
        "Open this notebook from do-dd-client (repo root or docs/notebooks/dirty/)."
    )


def _expected_python(repo_root: Path) -> Path:
    """Return the project venv interpreter."""
    for name in ("python", "python3"):
        candidate = repo_root / ".venv" / "bin" / name
        if candidate.is_file():
            return candidate.resolve()
    raise RuntimeError(f"No .venv interpreter under {repo_root}")


REPO_ROOT = _find_repo_root()
EXPECTED_PYTHON = _expected_python(REPO_ROOT)
ACTUAL_PYTHON = Path(sys.executable).resolve()

if ACTUAL_PYTHON != EXPECTED_PYTHON:
    raise RuntimeError(
        "Wrong notebook kernel for widget rendering.\n"
        f"  active:   {ACTUAL_PYTHON}\n"
        f"  expected: {EXPECTED_PYTHON}\n"
        "Run `make jupyter` and select kernel 'Python (do-dd-client)'."
    )

print("Kernel OK")
print(f"  python:     {ACTUAL_PYTHON}")
print(f"  anywidget:  {anywidget.__version__}")
print(f"  ipywidgets: {ipywidgets.__version__}")

In [ ]:
from pathlib import Path

from deeporigin.drug_discovery import BRD_DATA_DIR, Docking, Ligand, Pocket, Protein
from deeporigin.drug_discovery.docking_common import resolve_docking_box_geometry
from deeporigin.viz.molstar_html import MOLSTAR_JS_URL


def _find_repo_root() -> Path:
    """Return the CLI repo root whether cwd is repo root or docs/notebooks/dirty/."""
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "tests").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find do-dd-client repo root. Open this notebook from the repo "
        "(repo root or docs/notebooks/dirty/)."
    )


REPO_ROOT = _find_repo_root()
POCKET_FIXTURE = REPO_ROOT / "tests/fixtures/files/pocketfinder/pocket_1.pdb"

print(f"Repo root: {REPO_ROOT}")
print(f"Mol* bundle: {MOLSTAR_JS_URL}")
print(f"Pocket fixture: {POCKET_FIXTURE} ({'ok' if POCKET_FIXTURE.is_file() else 'missing'})")

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
if protein.id is None:
    protein.id = "notebook-demo-protein"

ligand = Ligand.from_sdf(BRD_DATA_DIR / "brd-2.sdf")
pocket = Pocket.from_pdb_file(str(POCKET_FIXTURE), name="pocket-1")
pocket.box_size_x = pocket.box_size_y = pocket.box_size_z = 15.0

box_center, box_size = resolve_docking_box_geometry(pocket)
print(f"Box center: {box_center}")
print(f"Box size:   {box_size}")

docking = Docking(protein=protein, pocket=pocket, ligand=ligand)
docking

## Interactive box viewer

1. Run the cell below — you should see a **Mol* iframe** (not a text repr).
2. Use **Settings** (gear) → **Docking Box** to rotate (sliders or canvas drag).
3. Click **Apply to notebook** in the bottom overlay.
4. Run the next cell to inspect `docking.rotation_deg`.


In [ ]:
handle = docking.show_box(interactive=True)
print(f"Bridge id: {handle.bridge_id}")

## Inspect committed rotation

In [ ]:
if docking.rotation_deg is None:
    print("No rotation committed yet — click Apply to notebook in the viewer.")
else:
    print(f"rotation_deg: {docking.rotation_deg}")

if handle.committed is not None:
    print(f"Last commit payload: {handle.committed}")

## Preview tool inputs (no execution)

Shows what `docking.run()` / `docking.start()` would send in the pocket block once
toolbox 3.4.32+ is deployed. Does not call the platform.

In [ ]:
params, _metadata = docking._build_tool_inputs()
params["pocket"]

## Run rotated docking (platform)

Apply a rotation above first. `run()` forwards the committed `rotation_deg` in the
pocket block. Poses are returned in the **original** protein frame (toolbox applies
the inverse transform after docking).

Optional: estimate cost before the billable run.

In [ ]:
if docking.rotation_deg is None:
    raise RuntimeError(
        "Commit a box rotation first: Settings → Docking Box → Apply to notebook."
    )

docking.run(quote=True)
docking.estimate

In [ ]:
print("Pocket sent to platform:")
print(docking._build_tool_inputs()[0]["pocket"])

poses = docking.run()
poses

## View docked poses

Static viewer — pose overlays are not supported in `interactive=True` mode.

In [ ]:
docking.show_box(poses=poses)